In [0]:
%run ./schemas

In [0]:
%run ./common

In [0]:
# Widgets supplied by driver; can also run independently for one repo/entity JSONL file.
import json, uuid, traceback
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.conf.set('spark.sql.session.timeZone','UTC')
for widget in ['batch_id','file_path','entity_type','repository','run_date']:
    dbutils.widgets.text(widget,'')
dbutils.widgets.dropdown('load_type','FULL',['FULL','INCREMENTAL'])
batch_id=dbutils.widgets.get('batch_id').strip()
file_path=dbutils.widgets.get('file_path').strip()
entity=dbutils.widgets.get('entity_type').strip()
repo=dbutils.widgets.get('repository').strip()
run_date=dbutils.widgets.get('run_date').strip()
load_type=dbutils.widgets.get('load_type')
start=now(); execution_id=str(uuid.uuid4()); count=0
try:
    date.fromisoformat(run_date)
    if not batch_id or not file_path or entity not in ENTITIES or repo not in REPOS:
        raise ValueError('Required: batch_id, file_path, entity_type, repository, run_date')
    if not file_path.startswith('/Volumes/github_health/raw/bronze_data/bronze/'):
        raise ValueError('Input must be from expected raw Bronze Volume')
    if not file_path.endswith('.jsonl'): raise ValueError('Input must be JSONL')
    if not __import__('pathlib').Path(file_path).is_file(): raise FileNotFoundError(file_path)
    ensure_entity_tables(spark)

    # STRICT explicit schema and permissive corrupt record capture. No inferSchema.
    raw=(spark.read.schema(RAW_SCHEMAS[entity]).option('mode','PERMISSIVE')
         .option('columnNameOfCorruptRecord','_corrupt_record').json(file_path))
    raw=raw.withColumn('_file_path',F.lit(file_path))  # Serverless: no DataFrame cache
    count=raw.count()

    if entity=='commits':
        raw_id=F.col('sha')
        updated=F.col('commit.author.date')
    else:
        raw_id=F.col('id').cast('string')
        updated=F.col('updated_at')

    # Protect against records loaded into wrong repository or issues containing PR objects.
    bad_reason=(F.when(F.col('_corrupt_record').isNotNull(),F.lit('BAD_JSON_OR_TYPE'))
      .when(raw_id.isNull() | (F.trim(raw_id)=='') ,F.lit('MISSING_PRIMARY_ID'))
      .when(F.col('_repo').isNotNull() & (F.col('_repo')!=repo),F.lit('REPOSITORY_MISMATCH')))
    if entity=='issues':
        bad_reason=bad_reason.when(F.col('pull_request').isNotNull(),F.lit('PULL_REQUEST_IN_ISSUES'))

    staged=(raw.withColumn('_reason',bad_reason)
        .withColumn('_entity_id',raw_id)
        .withColumn('_updated_str',updated))
    failures=staged.filter(F.col('_reason').isNotNull())

    q=(failures.select(
        F.sha2(F.concat_ws('|',F.lit(batch_id),F.lit(repo),F.lit(entity),
             F.coalesce(F.col('_corrupt_record'),F.to_json(F.struct(*[
                F.col(c) for c in RAW_SCHEMAS[entity].fieldNames() if c!='_corrupt_record'
             ])))),256).alias('quarantine_id'),
        F.lit(batch_id).alias('batch_id'),F.lit(repo).alias('repository'),
        F.lit(entity).alias('entity'),F.lit(file_path).alias('source_file'),
        F.coalesce(F.col('_corrupt_record'),F.to_json(F.struct(*[
           F.col(c) for c in RAW_SCHEMAS[entity].fieldNames() if c!='_corrupt_record'
        ]))).alias('raw_payload'),
        F.col('_reason').alias('error_type'),
        F.concat(F.lit('Failed Bronze validation: '),F.col('_reason')).alias('error_reason'),
        F.current_timestamp().alias('load_timestamp'),
        F.current_timestamp().alias('quarantined_at')
    ))
    quarantined=quarantine(spark,q,'bronze')

    valid=staged.filter(F.col('_reason').isNull())
    # Hash excludes run-specific metadata so unchanged replays are no-ops.
    business_cols={'commits':['sha','node_id','commit','author','committer','html_url'],
                   'issues':['id','number','node_id','title','state','created_at',
                             'updated_at','closed_at','user','labels','html_url'],
                   'pulls':['id','number','node_id','title','state','draft',
                            'created_at','updated_at','closed_at','merged_at',
                            'user','merged','html_url']}
    relevant_json=F.to_json(F.struct(*[F.col(c) for c in business_cols[entity]]))
    # Retain all explicitly selected typed source columns in Bronze; source JSONL
    # remains authoritative for fields not modeled in this phase.
    payload=F.to_json(F.struct(*[F.col(c) for c in RAW_SCHEMAS[entity].fieldNames()
                            if c!='_corrupt_record']))
    source=valid.select(
        F.concat_ws('|',F.lit(repo),F.col('_entity_id')).alias('record_key'),
        F.lit(repo).alias('repository'),F.lit(batch_id).alias('batch_id'),
        F.lit(file_path).alias('source_file'),
        F.expr('try_cast(_updated_str as timestamp)').alias('source_updated_at'),
        payload.alias('raw_payload'),
        F.sha2(relevant_json,256).alias('payload_hash'),
        F.current_timestamp().alias('load_timestamp'),
        F.expr('try_cast(_ingested_at as timestamp)').alias('ingested_at')
    )
    # If multiple rows with same ID appear, choose the newest update before MERGE.
    rank=Window.partitionBy('record_key').orderBy(F.col('source_updated_at').desc_nulls_last(),
                                                  F.col('payload_hash').desc())
    source=source.withColumn('_rn',F.row_number().over(rank)).filter('_rn=1').drop('_rn')
    written=source.count()
    upserted=merge_changed(spark,source,f'{CATALOG}.bronze.{entity}')
    print(f'BRONZE {repo}/{entity}: input={count} valid_keys={written} quarantined={quarantined}')
    log(spark,execution_id,batch_id,'raw_to_bronze','BRONZE',run_date,load_type,start,'SUCCESS',count,
        f'valid_keys={written}; quarantined={quarantined}',
        upserted=upserted, quarantined=quarantined)
except Exception:
    err=traceback.format_exc()
    try: log(spark,execution_id,batch_id,'raw_to_bronze','BRONZE',run_date or '1970-01-01',
             load_type,start,'FAILED',count,err)
    except Exception as e: print('LOGGING FAILED:',e)
    raise

dbutils.notebook.exit(json.dumps({'status':'SUCCESS','rows_processed':count,
                                   'valid_keys':written,'upserted':upserted,'quarantined':quarantined}))
